<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

# Green Thumbs-Up Superset (PACs)

This notebook is a test sample to upload all the CSV to firebase.

Sai, Lindsey and I still need to agree where the data will be uploaded (Firebase project).


# Imports

In [1]:
import pandas as pd
import re

# Load the Datasets

In [2]:
U = pd.read_csv("company_universe.csv", dtype=str)  # one column: TICKER
xwalk = pd.read_csv("company_ticker_crosswalk.csv", dtype=str)  # TICKER, COMPANY_NAME
pac = pd.read_csv("company_pac_donation_status.csv", dtype=str)  # CONNECTED_ORG_NM + flags

# Normalization

In [3]:
def norm_name(s: str) -> str:
    if s is None:
        return ""
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\b(inc|incorporated|corp|corporation|co|company|llc|ltd|plc|sa|ag|nv)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

xwalk["company_name_clean"] = xwalk["COMPANY_NAME"].map(norm_name)
pac["company_name_clean"] = pac["CONNECTED_ORG_NM"].map(norm_name)

# Collapse PAC status to one row per company name

In [4]:
def to_bool_series(s):
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .isin(["true", "1", "yes", "y", "t"])
    )
    
pac["has_pac"] = pac["has_pac"].astype(str).str.lower().isin(["true", "1", "yes"])
pac["has_pac_donations"] = pac["has_pac_donations"].astype(str).str.lower().isin(["true", "1", "yes"])

pac["pac_count"] = pd.to_numeric(pac.get("pac_count"), errors="coerce")
pac["active_pac_count"] = pd.to_numeric(pac.get("active_pac_count"), errors="coerce")
pac["total_pac_amount"] = pd.to_numeric(pac.get("total_pac_amount"), errors="coerce")

pac_name_level = (
    pac.groupby("company_name_clean", as_index=False)
       .agg(
           has_pac=("has_pac", "any"),
           has_pac_donations=("has_pac_donations", "any"),
           pac_count=("pac_count", "max"),
           active_pac_count=("active_pac_count", "max"),
           total_pac_amount=("total_pac_amount", "max"),
       )
)

# Map company name

In [5]:
pac_to_ticker = xwalk.merge(pac_name_level, on="company_name_clean", how="inner")

# Join the universe U

In [6]:
pac_ticker_level = (
    pac_to_ticker.groupby("TICKER", as_index=False)
                 .agg(
                     has_pac=("has_pac", "any"),
                     has_pac_donations=("has_pac_donations", "any"),
                     pac_count=("pac_count", "max"),
                     active_pac_count=("active_pac_count", "max"),
                     total_pac_amount=("total_pac_amount", "max"),
                 )
)

final = U.merge(
    pac_ticker_level[
        [
            "TICKER",
            "has_pac",
            "has_pac_donations",
            "pac_count",
            "active_pac_count",
            "total_pac_amount",
        ]
    ],
    on="TICKER",
    how="left",
)


# Add a clean status label

In [7]:
def status(row):
    if not row["has_pac"]:
        return "NO_PAC"
    if row["has_pac_donations"]:
        return "PAC_DONATED"
    return "PAC_NO_DONATIONS"

final["pac_status"] = final.apply(status, axis=1)

# Outputs

In [8]:
final.to_csv("company_pac_superset_U.csv", index=False)

# --- Sanity checks ---
print("Universe tickers:", len(final))
print(final["pac_status"].value_counts())
print("Tickers matched to PAC companies:", (final["has_pac"] == True).sum())

Universe tickers: 1250
pac_status
PAC_DONATED         1197
PAC_NO_DONATIONS      53
Name: count, dtype: int64
Tickers matched to PAC companies: 236
